# 10_bronze_transit_ingest
Generate a small batch of simulated transit telemetry and appends it to `bronze_events`. This is a placeholder source for the MVP and will later be replaced by HSL ingestion.

In [0]:
# Event schema (Bronze layer)
# event_id: unique event identifier
# event_time: original timestamp string from source
# event_time_ts: parsed event timestamp
# ingest_time_ts: ingestion timestamp
# entity_type: entity category (vehicle)
# entity_id: vehicle identifier
# metric: telemetry metric (delay_sec, occupancy)
# value: metric value
# unit: unit of measurement
# attrs: additional attributes (route_id, stop_id)
spark.sql("USE azure_streaming_mvp")

# Quick guard: ensure Bronze tables exists
spark.sql("SHOW TABLES").show(truncate=False)

+-------------------+-------------+-----------+
|database           |tableName    |isTemporary|
+-------------------+-------------+-----------+
|azure_streaming_mvp|bronze_events|false      |
+-------------------+-------------+-----------+



In [0]:
import uuid
import time, random
from pyspark.sql import functions as F

ROUTES = ["M1", "M2", "T1", "R10", "B1", "B2", "X3", "X7"]
TIME_SPAN_MINUTES = 60 # simulation window (minutes)

In [0]:
def make_event() -> dict:
    # Two telemetry metrics in the MVP:
    # delay_sec -> vehicle delay in seconds (negative means early arrival)
    # occupancy -> estimated passenger load as percentage of capacity
    metric = random.choice(["delay_sec", "occupancy"])
    # Simulatd value ranges:
    # delay_sec: -30 to 600 seconds (slightly early to up to 10 min delay)
    # occupancy: 0-80% passenger load
    value = random.randint(-30, 600) if metric == "delay_sec" else random.randint(0, 80)

    # Randomize event time within last TIME_SPAN_MINUTES
    offset_sec = random.randint(0, TIME_SPAN_MINUTES * 60)
    event_epoch = int(time.time()) - offset_sec # UTC epoch seconds
    event_time_raw = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(event_epoch))

    return {
        "event_id": str(uuid.uuid4()),
        "event_time_raw": event_time_raw,
        "source": "sim_transit",
        "entity_type": "vehicle",
        "entity_id": f"veh_{random.randint(100, 130)}",
        "metric": metric,
        "value": float(value),
        "unit": "sec" if metric == "delay_sec" else "pct",
        "attrs": {"route_id": random.choice(ROUTES),
                  "stop_id": str(random.randint(1, 60))}
    }

In [0]:
def ingest_batch(batch_id: int, n: int = 200, dry_run: bool = False) -> None:
    """
    Generate a simulated batch of transit telemetry events
    and append them to the Bronze table.

    Parameters
    ----------
    batch_id: int
       Identifier for the simulated batch.
    n: int
       Number of events to generate

    Notes
    -----
    This placeholder ingestion simulates a streaming source.
    In production this layer will be replaced by HSL API ingestion.
    """
    # Generate simulated events
    rows = [make_event() for _ in range(n)]
    # Create Spark DataFrame
    df = spark.createDataFrame(rows)    

    # Parse event time and add ingest timestamp
    df2 = (df
           .withColumn("event_time_ts", F.to_timestamp("event_time_raw", "yyyy-MM-dd'T'HH:mm:ss'Z'"))
           .withColumn("ingest_time_ts", F.current_timestamp())
    )

    if dry_run:
       print("DRY RUN - no data written to bronze_events")
       display(df2.limit(10))
       return

    # Append to Bronze table
    (df2.write.mode("append").saveAsTable("bronze_events"))
    print(f"Batch {batch_id} appended to bronze_events: {n} rows")

In [0]:
ingest_batch(0, 200)   # When debugging, run ingest_batch(0, 200, dry_run=True)

Batch 0 appended to bronze_events: 200 rows


### Data quality visibility checks

The Bronze layer keeps raw telemetry events unchanged.
These simple checks help surface unrealistic values without modifying the data.

In [0]:
%sql
-- Row Count (sim_transit)
SELECT COUNT(*) AS n
FROM bronze_events
WHERE source = "sim_transit";

n
200


In [0]:
%sql
-- Latest event_time (sim_transit)
SELECT MAX(event_time_ts) AS latest_event_time
FROM bronze_events
WHERE source = "sim_transit";

latest_event_time
2026-03-08T15:43:49.000Z


In [0]:
display(
    spark.table("bronze_events")
    .where(F.col("source") == "sim_transit")
    .orderBy(F.col("ingest_time_ts").desc())
    .limit(10)
)

event_id,event_time_raw,source,entity_type,entity_id,metric,value,unit,attrs,event_time_ts,ingest_time_ts
945bf560-644f-43f8-afb7-ce960d6b3eec,2026-03-08T14:46:48Z,sim_transit,vehicle,veh_123,delay_sec,-23.0,sec,"Map(route_id -> M1, stop_id -> 48)",2026-03-08T14:46:48.000Z,2026-03-08T15:44:55.161Z
6ebdc6b7-d60d-41cf-b8e6-51929f1e2c51,2026-03-08T15:01:17Z,sim_transit,vehicle,veh_106,occupancy,72.0,pct,"Map(route_id -> B1, stop_id -> 55)",2026-03-08T15:01:17.000Z,2026-03-08T15:44:55.161Z
72ddc59f-4674-41b2-ae0a-79eba1541c83,2026-03-08T14:47:23Z,sim_transit,vehicle,veh_120,delay_sec,298.0,sec,"Map(route_id -> X7, stop_id -> 56)",2026-03-08T14:47:23.000Z,2026-03-08T15:44:55.161Z
8521cb46-a27e-49df-ad23-4e7698870e7e,2026-03-08T15:17:07Z,sim_transit,vehicle,veh_124,occupancy,79.0,pct,"Map(route_id -> B1, stop_id -> 41)",2026-03-08T15:17:07.000Z,2026-03-08T15:44:55.161Z
2e29f801-f164-489e-b6a1-918c80b37f2b,2026-03-08T15:17:42Z,sim_transit,vehicle,veh_117,occupancy,39.0,pct,"Map(route_id -> B2, stop_id -> 36)",2026-03-08T15:17:42.000Z,2026-03-08T15:44:55.161Z
3f377b78-4977-4744-a6fe-f60ca6ee2bc9,2026-03-08T14:55:53Z,sim_transit,vehicle,veh_111,delay_sec,170.0,sec,"Map(route_id -> M1, stop_id -> 52)",2026-03-08T14:55:53.000Z,2026-03-08T15:44:55.161Z
3a6c46f8-3615-4955-bbd5-fa79dc27b9ce,2026-03-08T14:47:19Z,sim_transit,vehicle,veh_124,occupancy,9.0,pct,"Map(route_id -> B1, stop_id -> 4)",2026-03-08T14:47:19.000Z,2026-03-08T15:44:55.161Z
0e6b2d73-a9d4-4a1f-a676-f1bda7297283,2026-03-08T15:29:31Z,sim_transit,vehicle,veh_129,delay_sec,157.0,sec,"Map(route_id -> M1, stop_id -> 41)",2026-03-08T15:29:31.000Z,2026-03-08T15:44:55.161Z
e8fffe81-7d5e-4033-bd94-5ba414d275f9,2026-03-08T14:59:54Z,sim_transit,vehicle,veh_103,occupancy,75.0,pct,"Map(route_id -> B2, stop_id -> 27)",2026-03-08T14:59:54.000Z,2026-03-08T15:44:55.161Z
709acf6a-cda4-4498-9fa5-26975524c0f6,2026-03-08T15:05:46Z,sim_transit,vehicle,veh_100,occupancy,32.0,pct,"Map(route_id -> B1, stop_id -> 42)",2026-03-08T15:05:46.000Z,2026-03-08T15:44:55.161Z


In [0]:
%sql
-- Check for duplicate event_id
SELECT event_id, COUNT(*)
FROM bronze_events
GROUP BY event_id
HAVING COUNT(*) > 1;

event_id,COUNT(*)


In [0]:
%sql
-- Sanity check: unrealistic delay values
SELECT *
FROM bronze_events
WHERE metric = 'delay_sec'
AND (value < -120 OR value > 3600)
LIMIT 20

event_id,event_time_raw,source,entity_type,entity_id,metric,value,unit,attrs,event_time_ts,ingest_time_ts


In [0]:
%sql
-- Sanity check: occupancy should be between 0 and 100
SELECT *
FROM bronze_events
WHERE metric = 'occupancy'
AND (value < 0 OR value > 100)
LIMIT 20

event_id,event_time_raw,source,entity_type,entity_id,metric,value,unit,attrs,event_time_ts,ingest_time_ts


In [0]:
# ---- Notebook completion signal ----
dbutils.notebook.exit("OK")